# Docling — Document Layout Analysis report

Notebook này chạy inference trên toàn bộ ảnh trong split DocLayNet local (hiện 1,928 ảnh), tính Precision/Recall/F1 bằng `src/common/metrics.py`, rồi tạo và hiển thị tối đa 20 ảnh Ground Truth so với prediction bằng `visualization/scripts/visualize_predictions.py`.

Chạy lần lượt từ trên xuống. Lần inference đầu tiên có thể tải model Docling từ Hugging Face. Trên Windows, cell inference đặt `HF_HUB_DISABLE_SYMLINKS=1` để dùng bản sao file trong cache nếu symbolic link không được phép.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import json

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "methods").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "methods").is_dir():
    raise FileNotFoundError("Không tìm thấy thư mục gốc repository; hãy mở notebook từ project DO-AN-261.")

METHOD = "docling"
DATASET = "doclaynet"
SPLIT = "train"
VISUALIZATION_LIMIT = 20
IOU_THRESHOLD = 0.5
GT_PATH = ROOT / "datasets" / DATASET / "processed" / "annotations" / f"{SPLIT}.json"
if not GT_PATH.is_file():
    raise FileNotFoundError(f"Không tìm thấy ground truth: {GT_PATH}")
with GT_PATH.open(encoding="utf-8") as f:
    EXPECTED_IMAGE_IDS = {image["id"] for image in json.load(f)["images"]}
PREDICTIONS_DIR = ROOT / "outputs" / "predictions" / METHOD / DATASET
FIGURES_DIR = ROOT / "outputs" / "reports" / f"{METHOD}_{DATASET}"
print(f"Project: {ROOT}\nMethod: {METHOD} | Dataset: {DATASET}/{SPLIT} | Images to evaluate: {len(EXPECTED_IMAGE_IDS)} | Visualization limit: {VISUALIZATION_LIMIT}")

## 1. Chạy inference

Lệnh gọi source inference chính với `--skip-existing`, nên notebook có thể tiếp tục một lượt chạy bị ngắt mà không tính lại prediction đã có.

In [ ]:
inference_script = ROOT / "methods" / METHOD / "src" / "run_inference.py"
if not inference_script.is_file():
    raise FileNotFoundError(f"Chưa có source inference: {inference_script}")

env = os.environ.copy()
env["HF_HUB_DISABLE_SYMLINKS"] = "1"
command = [sys.executable, str(inference_script), "--dataset", DATASET, "--limit", "0", "--skip-existing"]
print("Running:", " ".join(command))
subprocess.run(command, cwd=ROOT, env=env, check=True)
prediction_files = sorted(PREDICTIONS_DIR.glob("*.json"))
print(f"Prediction files available: {len(prediction_files)} ({PREDICTIONS_DIR})")

## 2. Đánh giá metric

Dùng greedy matching theo IoU của project trên toàn bộ ảnh trong ground truth. Strict yêu cầu khớp cả vị trí và nhãn; loose chỉ khớp vị trí để xem lỗi localization so với lỗi phân loại. Cell kiểm tra thiếu prediction và giữ các ảnh không có ground-truth box trong phép đánh giá.

In [ ]:
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "src"))
from common.metrics import evaluate
from scripts.evaluate_predictions import load_gt, load_predictions
import pandas as pd
from IPython.display import display

gt_path = ROOT / "datasets" / DATASET / "processed" / "annotations" / f"{SPLIT}.json"
if not gt_path.is_file():
    raise FileNotFoundError(f"Không tìm thấy ground truth: {gt_path}")
gt_by_image, category_names = load_gt(gt_path)
gt_by_image = {image_id: gt_by_image.get(image_id, []) for image_id in EXPECTED_IMAGE_IDS}
pred_by_image = load_predictions(PREDICTIONS_DIR)
missing_predictions = EXPECTED_IMAGE_IDS - pred_by_image.keys()
if missing_predictions:
    raise RuntimeError(f"Thiếu {len(missing_predictions)} prediction trên {len(EXPECTED_IMAGE_IDS)} ảnh; chạy lại cell inference để tiếp tục.")
gt_subset = {image_id: boxes for image_id, boxes in gt_by_image.items() if image_id in EXPECTED_IMAGE_IDS}
strict = evaluate(gt_subset, pred_by_image, category_names, IOU_THRESHOLD, require_same_category=True)
loose = evaluate(gt_subset, pred_by_image, category_names, IOU_THRESHOLD, require_same_category=False)
print(f"Đánh giá {len(gt_subset)} ảnh; IoU >= {IOU_THRESHOLD}")

def overall_row(name, result):
    values = result["overall"]
    return {"Metric": name, "Precision": values["precision"], "Recall": values["recall"], "F1": values["f1"], "TP": values["tp"], "FP": values["fp"], "FN": values["fn"], "Mean IoU (matched)": values["mean_iou_matched"]}

display(pd.DataFrame([overall_row("Strict (vị trí + nhãn)", strict), overall_row("Loose (chỉ vị trí)", loose)]).set_index("Metric").round(3))
per_category = []
for category_id, values in strict["per_category"].items():
    per_category.append({"ID": category_id, "Category": values["name"], "Support": values["support"], "TP": values["tp"], "FP": values["fp"], "FN": values["fn"], "Precision": values["precision"], "Recall": values["recall"], "F1": values["f1"]})
display(pd.DataFrame(per_category).sort_values("Support", ascending=False).set_index("Category").round(3))

## 3. Ground Truth vs. prediction

Gọi visualizer dùng chung trong repository và hiển thị một ảnh đối chiếu.

In [ ]:
from IPython.display import Image as DisplayImage, display

visualizer = ROOT / "visualization" / "scripts" / "visualize_predictions.py"
command = [sys.executable, str(visualizer), "--method", METHOD, "--dataset", DATASET, "--split", SPLIT, "--limit", str(VISUALIZATION_LIMIT), "--output-dir", str(FIGURES_DIR)]
subprocess.run(command, cwd=ROOT, check=True)
figure_files = sorted(FIGURES_DIR.glob("*.png"), key=lambda path: int(path.stem) if path.stem.isdigit() else path.stem)
if figure_files:
    print(figure_files[0])
    display(DisplayImage(filename=str(figure_files[0])))
else:
    print(f"Visualizer không tạo ảnh nào trong {FIGURES_DIR}")

## Files đầu ra

- Predictions: `outputs/predictions/docling/doclaynet/`
- Ground Truth và predictions side-by-side: `outputs/reports/docling_doclaynet/`
- Ground Truth: `datasets/doclaynet/processed/annotations/train.json`